# Notebook 05 — Final Scorecard

**One place to see every comparable number and every important plot** from the project so far. This notebook does not re-derive anything new; it just loads the frozen artifacts produced by Notebooks 02/03/04 and renders them side-by-side so you can read the whole research arc in one pass.

Sections:

1. Master scorecard table (all strategies, all phases)
2. Equity curves — v1 (Notebook-03 final) vs v2 (Phase-12 final)
3. Drawdown comparison
4. Monthly Sharpe — v1 vs v2
5. Rolling 21-day Sharpe — stability
6. Cost sensitivity — v1 full-sample vs v2 TEST
7. Bootstrap Sharpe distribution — v1 full vs v2 TEST
8. Proxy robustness (XLK / QQQ / SPY)
9. Drop-top-k-month stress test
10. Diagnostic: dispersion concentration by month
11. Turnover distribution — the cost story in one picture
12. March 2026 freeze — the closest thing to a live read
13. One-sentence verdict

In [ ]:
import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
VAL = pathlib.Path('../data_qtc/validation')

phase12 = json.loads((VAL / 'phase12_summary.json').read_text())
v1_full_summary = json.loads((VAL / 'summary.json').read_text())
bt_v1 = pd.read_parquet(VAL / 'phase12_bt_v1.parquet')
bt_v2 = pd.read_parquet(VAL / 'phase12_bt_v2.parquet')
rs21 = pd.read_parquet(VAL / 'rolling_sharpe_21d.parquet')
mon_v1 = pd.read_csv(VAL / 'monthly_sharpe.csv')
mon_v2 = pd.read_csv(VAL / 'phase12_monthly_test.csv')
diag = pd.read_csv(VAL / 'monthly_diagnostics.csv', index_col=0)
proxy_v1 = pd.read_csv(VAL / 'proxy_robustness.csv')
proxy_v2 = pd.read_csv(VAL / 'phase12_proxy_robustness.csv')
cost_v1 = pd.read_csv(VAL / 'cost_stress_full.csv')
cost_v2 = pd.read_csv(VAL / 'phase12_cost_stress_test.csv')

print('Loaded artifacts:')
print(f'  bt_v1 post-warmup bars: {len(bt_v1):,}  range: {bt_v1.index.min()} → {bt_v1.index.max()}')
print(f'  bt_v2 post-warmup bars: {len(bt_v2):,}')
print(f'  phase12_summary keys: {list(phase12.keys())}')

## 1. Master scorecard — every strategy we can honestly compare

Numbers below come from RESEARCH.md (Act I + Act II frozen values) and `phase12_summary.json` (Act III live-loaded). All Sharpes are **annualised** at 1 bp/side cost unless noted. MaxDD is **percent**.

In [ ]:
rows = [
    # act, stage, signal, window, sharpe, ann_pct, maxdd_pct, note
    ('I',   'B0 naive MA',                     'time-series momentum (MA cross)', 'NVDA OOS 4mo',  -3.95, -139.0, -191.0, 'placebo baseline'),
    ('I',   'B1 + execution guardrails',       'time-series momentum',            'NVDA OOS 4mo',  -3.18, -104.0, -145.0, 'reduces bleed'),
    ('I',   'M1–M4 four-layer (alpha/regime/risk/exec)', 'time-series momentum',   'NVDA OOS 4mo',  -3.0,   -52.0,  -67.0, 'less-lose, not more-win'),
    ('I',   'M5 + triple-barrier meta-label',  'time-series momentum (filtered)', 'NVDA OOS 4mo',   0.35,   0.67,   -0.78, '52 trades → wide CI'),
    ('II',  'A naive xsec reversal',           'xsec past-return reversal',       'OOS 4mo',       -7.45, -158.0,  -45.0, 'cost eats gross'),
    ('II',  'C + sparse state machine',        'xsec residual reversal',          'OOS 4mo',        0.47,   9.6,  -11.2, '+10.46 Sharpe lift'),
    ('II',  'E Notebook-03 final (dispersion gate)', 'xsec residual reversal',    'OOS 4mo (narrow)', 2.72,  43.9,  -5.6, 'narrow window'),
    ('III', 'v1 replay on full sample',        'xsec residual reversal',          'Full 11mo',      phase12['v1_full']['sharpe'], phase12['v1_full']['ann_pct'], phase12['v1_full']['maxdd_pct'], 'the honest v1'),
    ('III', 'v2 Phase-12 (+breadth +per-stock)', 'xsec residual reversal',        'Full 11mo',      phase12['v2_full']['sharpe'], phase12['v2_full']['ann_pct'], phase12['v2_full']['maxdd_pct'], 'Act III delivery'),
    ('III', 'v2 TEST window',                  'xsec residual reversal',          'Aug 2025–Mar 2026', phase12['v2_test']['sharpe'], phase12['v2_test']['ann_pct'], phase12['v2_test']['maxdd_pct'], 'train/test split'),
    ('III', 'v2 March 2026 freeze',            'xsec residual reversal',          'Mar 2026 only',  phase12['v2_march_2026_freeze']['sharpe'], phase12['v2_march_2026_freeze']['ann_pct'], phase12['v2_march_2026_freeze']['maxdd_pct'], 'closest to live'),
]
score = pd.DataFrame(rows, columns=['Act', 'Strategy', 'Primary signal', 'Evaluation window', 'Sharpe', 'Ann %', 'MaxDD %', 'Note'])
score.style.format({'Sharpe': '{:+.2f}', 'Ann %': '{:+.1f}', 'MaxDD %': '{:+.1f}'})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
colors = {'I': '#8b8b8b', 'II': '#4c78a8', 'III': '#2d8659'}
bar_colors = [colors[a] for a in score['Act']]
ax.barh(range(len(score)), score['Sharpe'], color=bar_colors)
ax.set_yticks(range(len(score)))
ax.set_yticklabels(score['Strategy'], fontsize=9)
ax.invert_yaxis()
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('Sharpe (annualised, 1bp/side)')
ax.set_title('All comparable strategies — Sharpe (grey=Act I, blue=Act II, green=Act III)')
for i, (s, n) in enumerate(zip(score['Sharpe'], score['Note'])):
    ax.text(s + (0.1 if s >= 0 else -0.1), i, f'{s:+.2f}', ha='left' if s >= 0 else 'right', va='center', fontsize=8)
plt.tight_layout(); plt.show()

## 2. Equity curves — v1 vs v2 (post-warmup full sample)

Both curves use exactly the same wide return matrix, the same entry threshold, the same holding rule. The only difference between them is the regime gate: v1 uses the single `dispersion_regime_filter`; v2 adds `breadth_regime_filter` and `per_stock_vol_mask`. Shaded region is the honest TRAIN-window portion (2025-05 → 2025-07 post-warmup).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(bt_v1.index, bt_v1['equity'] * 100, label='v1 Notebook-03 final', color='#4c78a8', lw=1.2)
ax.plot(bt_v2.index, bt_v2['equity'] * 100, label='v2 Phase-12 final',    color='#2d8659', lw=1.6)
train_end = pd.Timestamp('2025-08-01', tz='UTC')
ax.axvspan(bt_v2.index.min(), train_end, color='#cccccc', alpha=0.25, label='TRAIN (design-visible) window')
ax.axhline(0, color='k', lw=0.7)
ax.set_ylabel('cumulative net PnL (%)')
ax.set_title('Equity curves on full post-warmup sample — v1 vs v2')
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

## 3. Drawdown comparison

The single most important plot for this comparison. v1 was not a bad strategy; it had a bad drawdown profile that was hidden by Notebook 03's narrow window. v2 cuts MaxDD from **−14.1%** to **−4.8%** while raising the Sharpe.

In [ ]:
def drawdown(equity):
    eq = equity.copy()
    peak = eq.cummax()
    return (eq - peak)
dd_v1 = drawdown(bt_v1['equity']) * 100
dd_v2 = drawdown(bt_v2['equity']) * 100
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.fill_between(dd_v1.index, dd_v1, 0, color='#4c78a8', alpha=0.45, label=f'v1 (min {dd_v1.min():.1f}%)')
ax.fill_between(dd_v2.index, dd_v2, 0, color='#2d8659', alpha=0.65, label=f'v2 (min {dd_v2.min():.1f}%)')
ax.axhline(0, color='k', lw=0.7)
ax.set_ylabel('drawdown (%)')
ax.set_title('Drawdown comparison — v1 vs v2 (full post-warmup sample)')
ax.legend(loc='lower left')
plt.tight_layout(); plt.show()

## 4. Monthly Sharpe — where the gap lives

v1 and v2 side-by-side by calendar month. The three red bars in v1 (Sep/Oct/Nov 2025) are the exact months Phase-12 was designed to fix. v2 turns all three positive.

In [ ]:
mon_v1_s = mon_v1.set_index('ym')['sharpe']
mon_v2_s = mon_v2.set_index(mon_v2.columns[0])['sharpe']
months = sorted(set(mon_v1_s.index) | set(mon_v2_s.index))
df_m = pd.DataFrame({'v1': mon_v1_s.reindex(months), 'v2': mon_v2_s.reindex(months)})
df_m = df_m.fillna(0.0)

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(df_m))
w = 0.4
ax.bar(x - w/2, df_m['v1'], width=w, label='v1', color=['#cc3333' if v < 0 else '#4c78a8' for v in df_m['v1']])
ax.bar(x + w/2, df_m['v2'], width=w, label='v2', color=['#cc3333' if v < 0 else '#2d8659' for v in df_m['v2']])
ax.axhline(0, color='k', lw=0.7)
ax.set_xticks(x); ax.set_xticklabels(df_m.index, rotation=45, fontsize=9)
ax.set_ylabel('monthly Sharpe'); ax.set_title('Monthly Sharpe — v1 vs v2')
ax.legend()
plt.tight_layout(); plt.show()
df_m.round(2)

## 5. Rolling 21-day Sharpe — stability

In [ ]:
W = 21 * 390
rs_v1 = (bt_v1['net_pnl'].rolling(W).mean() / bt_v1['net_pnl'].rolling(W).std(ddof=0)) * np.sqrt(390 * 252)
rs_v2 = (bt_v2['net_pnl'].rolling(W).mean() / bt_v2['net_pnl'].rolling(W).std(ddof=0)) * np.sqrt(390 * 252)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(rs_v1.index, rs_v1, color='#4c78a8', lw=1.0, alpha=0.9, label='v1 rolling 21d Sharpe')
ax.plot(rs_v2.index, rs_v2, color='#2d8659', lw=1.3, label='v2 rolling 21d Sharpe')
ax.axhline(0, color='k', lw=0.6); ax.axhline(1, color='#999', ls='--', lw=0.6)
ax.set_ylabel('rolling 21-day Sharpe')
ax.set_title('Rolling 21-day Sharpe — v1 vs v2')
ax.legend()
plt.tight_layout(); plt.show()

pd.DataFrame({
    'p05':  [rs_v1.quantile(0.05), rs_v2.quantile(0.05)],
    'p25':  [rs_v1.quantile(0.25), rs_v2.quantile(0.25)],
    'p50':  [rs_v1.quantile(0.50), rs_v2.quantile(0.50)],
    'p75':  [rs_v1.quantile(0.75), rs_v2.quantile(0.75)],
    'p95':  [rs_v1.quantile(0.95), rs_v2.quantile(0.95)],
    'frac > 0': [(rs_v1.dropna() > 0).mean(), (rs_v2.dropna() > 0).mean()],
    'frac > 1': [(rs_v1.dropna() > 1).mean(), (rs_v2.dropna() > 1).mean()],
}, index=['v1', 'v2']).round(2)

## 6. Cost sensitivity — v1 full-sample vs v2 TEST

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cost_v1['cost_bps'], cost_v1['sharpe'], marker='o', color='#4c78a8', label='v1 — full 11mo post-warmup')
ax.plot(cost_v2['cost_bps'], cost_v2['sharpe'], marker='s', color='#2d8659', label='v2 — TEST window (8mo)')
ax.axhline(0, color='k', lw=0.7)
ax.axvline(1.0, color='#999', ls='--', lw=0.7, label='QTC IEX realistic ~1bp')
ax.axvline(2.0, color='#bbb', ls=':', lw=0.7, label='adverse slippage ~2bp')
ax.set_xlabel('cost (bp/side)'); ax.set_ylabel('Sharpe')
ax.set_title('Cost sensitivity — v1 vs v2')
ax.legend()
plt.tight_layout(); plt.show()
pd.DataFrame({
    'cost_bps': cost_v1['cost_bps'],
    'v1 full Sharpe': cost_v1['sharpe'].round(2),
    'v2 TEST Sharpe': cost_v2.set_index('cost_bps').reindex(cost_v1['cost_bps'])['sharpe'].round(2).values,
})

## 7. Bootstrap Sharpe distribution — how much did v2 move the needle?

2000 stationary block-bootstrap draws, block length = 2 trading days = 780 minute bars. For v1 we bootstrap on the full 11-month post-warmup; for v2 we bootstrap on the 8-month TEST window.

In [ ]:
def block_bootstrap_sharpe(pnl, block_len=2*390, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    a = pnl.to_numpy()
    n_bl = len(a) // block_len
    out = []
    for _ in range(n):
        starts = rng.integers(0, len(a) - block_len + 1, size=n_bl)
        s = np.concatenate([a[s:s+block_len] for s in starts])
        sd = s.std(ddof=0)
        out.append(0.0 if sd == 0 else s.mean() / sd * np.sqrt(390*252))
    return np.array(out)

sh_v1 = block_bootstrap_sharpe(bt_v1['net_pnl'])
ts_ny = bt_v2.index.tz_convert('America/New_York')
bt_v2_test = bt_v2.loc[ts_ny >= pd.Timestamp('2025-08-01', tz='America/New_York')]
sh_v2 = block_bootstrap_sharpe(bt_v2_test['net_pnl'])

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sh_v1, bins=50, color='#4c78a8', alpha=0.6, label=f'v1 full — median {np.median(sh_v1):+.2f}, P(>1)={(sh_v1>1).mean():.0%}')
ax.hist(sh_v2, bins=50, color='#2d8659', alpha=0.7, label=f'v2 TEST — median {np.median(sh_v2):+.2f}, P(>1)={(sh_v2>1).mean():.0%}')
ax.axvline(0, color='k', lw=0.7); ax.axvline(1, color='#999', ls='--', lw=0.7)
ax.set_xlabel('bootstrap Sharpe'); ax.set_title('Block-bootstrap Sharpe distribution (2000 draws, 2-day blocks)')
ax.legend()
plt.tight_layout(); plt.show()

def boot_stats(a):
    return {
        'median': np.median(a), 'p05': np.quantile(a, 0.05), 'p25': np.quantile(a, 0.25),
        'p75': np.quantile(a, 0.75), 'p95': np.quantile(a, 0.95),
        'P(>0)': (a > 0).mean(), 'P(>1)': (a > 1).mean(),
    }
pd.DataFrame({'v1 full': boot_stats(sh_v1), 'v2 TEST': boot_stats(sh_v2)}).round(2)

## 8. Proxy robustness (XLK / QQQ / SPY)

Replacing the residualisation proxy. Negative Sharpes for QQQ/SPY are **expected**, not a fragility — see RESEARCH.md §13.5 for the theoretical argument (QQQ contains the 4 target names at ~40% weight, SPY has wrong factor composition).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(proxy_v1))
w = 0.4
ax.bar(x - w/2, proxy_v1['sharpe'], width=w, color='#4c78a8', label='v1 full')
ax.bar(x + w/2, proxy_v2['sharpe'], width=w, color='#2d8659', label='v2 full')
ax.set_xticks(x); ax.set_xticklabels(proxy_v1['proxy'])
ax.axhline(0, color='k', lw=0.7)
ax.set_ylabel('Sharpe'); ax.set_title('Alt-proxy robustness — v1 vs v2 (full-sample post-warmup)')
ax.legend()
plt.tight_layout(); plt.show()
pd.merge(proxy_v1[['proxy', 'sharpe', 'ann_return', 'max_dd']].rename(columns={'sharpe': 'v1 Sharpe', 'ann_return': 'v1 ann', 'max_dd': 'v1 MaxDD'}),
         proxy_v2[['proxy', 'sharpe', 'ann_pct', 'maxdd_pct']].rename(columns={'sharpe': 'v2 Sharpe', 'ann_pct': 'v2 ann%', 'maxdd_pct': 'v2 MaxDD%'}),
         on='proxy')

## 9. Drop-top-k stress test

"If you delete the best k months, what's left?" For a real edge, Sharpe should not collapse after removing the top 1 – 3 months. For v1 it does; for v2 it doesn't.

In [ ]:
def sharpe_s(x):
    s = x.std(ddof=0)
    return 0.0 if s == 0 else float(x.mean() / s * np.sqrt(390 * 252))

def drop_top_k(bt):
    ym = bt.index.tz_convert('America/New_York').to_period('M')
    monthly_sum = bt.groupby(ym)['net_pnl'].sum().sort_values(ascending=False)
    res = []
    for k in range(0, 5):
        to_drop = set(monthly_sum.head(k).index.astype(str))
        mask = ~ym.astype(str).isin(to_drop)
        res.append((k, sharpe_s(bt.loc[mask, 'net_pnl'])))
    return pd.DataFrame(res, columns=['k dropped', 'Sharpe'])

d_v1 = drop_top_k(bt_v1)
d_v2 = drop_top_k(bt_v2_test)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(d_v1['k dropped'], d_v1['Sharpe'], marker='o', color='#4c78a8', label='v1 full')
ax.plot(d_v2['k dropped'], d_v2['Sharpe'], marker='s', color='#2d8659', label='v2 TEST')
ax.axhline(0, color='k', lw=0.7); ax.axhline(1, color='#999', ls='--', lw=0.7)
ax.set_xlabel('k best months removed'); ax.set_ylabel('Sharpe')
ax.set_title('Drop-top-k-month stress test')
ax.legend()
plt.tight_layout(); plt.show()
pd.concat([d_v1.set_index('k dropped').rename(columns={'Sharpe': 'v1'}),
           d_v2.set_index('k dropped').rename(columns={'Sharpe': 'v2'})], axis=1).round(2)

## 10. Diagnostic — dispersion concentration by month

The single plot that justifies Iteration F. Each month is a point. Concentration = `std_stock(rolling_std) / mean_stock(rolling_std)`: high means *one stock is dominating dispersion*. Notice Sep/Oct 2025 sit well above the rest, and they are exactly v1's worst months.

In [ ]:
dd = diag.dropna(subset=['gate_on_frac'])
mon_v1_lookup = mon_v1.set_index('ym')['sharpe']
dd = dd.copy()
dd['v1 Sharpe'] = dd.index.map(mon_v1_lookup)
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(dd['disp_concentration'], dd['v1 Sharpe'].fillna(0), c=dd['v1 Sharpe'].fillna(0), cmap='RdYlGn', s=110, edgecolor='k')
for i, m in enumerate(dd.index):
    ax.annotate(m, (dd['disp_concentration'].iloc[i], dd['v1 Sharpe'].fillna(0).iloc[i]), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.axvline(dd['disp_concentration'].median(), color='#999', ls='--', lw=0.7, label='concentration median (60d)')
ax.axhline(0, color='k', lw=0.7)
ax.set_xlabel('dispersion concentration (higher = 1 stock dominates)')
ax.set_ylabel('v1 monthly Sharpe')
ax.set_title('Where v1 loses — high-concentration months')
plt.colorbar(sc, ax=ax, label='v1 Sharpe')
ax.legend()
plt.tight_layout(); plt.show()

## 11. Turnover — the cost story in one picture

v2 trades about a third as much as v1, which is *why* the break-even cost moved from ~1.4 bp to ~3.5 bp.

In [ ]:
tv1 = bt_v1['turnover'][bt_v1['turnover'] > 0]
tv2 = bt_v2['turnover'][bt_v2['turnover'] > 0]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(tv1, bins=40, color='#4c78a8', alpha=0.7, label=f'v1 — mean {tv1.mean():.4f}')
ax[0].hist(tv2, bins=40, color='#2d8659', alpha=0.7, label=f'v2 — mean {tv2.mean():.4f}')
ax[0].set_xlabel('per-bar turnover (|Δw|)'); ax[0].set_ylabel('count'); ax[0].set_title('Per-bar turnover distribution')
ax[0].legend()

def daily_turnover(bt):
    ts = bt.index.tz_convert('America/New_York').date
    return pd.Series(bt['turnover'].values, index=pd.DatetimeIndex(ts)).groupby(level=0).sum()
d1 = daily_turnover(bt_v1); d2 = daily_turnover(bt_v2)
ax[1].plot(d1.index, d1, color='#4c78a8', lw=0.9, alpha=0.8, label=f'v1 daily avg {d1.mean():.2f}x gross')
ax[1].plot(d2.index, d2, color='#2d8659', lw=0.9, label=f'v2 daily avg {d2.mean():.2f}x gross')
ax[1].set_ylabel('daily sum |Δw| (≈ book turns/day)'); ax[1].set_title('Daily turnover over time')
ax[1].legend()
plt.tight_layout(); plt.show()

## 12. Trade count & cost accounting

Count how many **position entries** the strategy produces (each time a stock goes from flat to long or short is one entry; each entry is one round-trip trade), and how much **dollar cost** gets paid, at different cost assumptions. This is the single most important operational sanity check: the whole reason Phase-12 works is because it trades *less*.

- Turnover = sum over time of `|Δweight|` per bar, summed across stocks.
- Total cost (bp × gross_capital) = `total_turnover × cost_bps / 10 000` of one unit of gross book.
- Cost as % of gross PnL tells us how much of the raw edge survives friction.

Both strategies are evaluated on the full 11-month post-warmup sample. v2 is additionally shown on its 8-month TEST window.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.xsec_strategy import build_final_strategy, build_final_strategy_v2

panel = pd.read_parquet('../data_qtc/combined/research_panel.parquet')
TARGETS = ['AAPL', 'AMD', 'MSFT', 'NVDA']
WARMUP = 60 * 390

v1 = build_final_strategy(panel, TARGETS, proxy_symbol='XLK')
v2 = build_final_strategy_v2(panel, TARGETS, proxy_symbol='XLK')
w1 = v1['weights_final'].iloc[WARMUP:]
w2 = v2['weights_final'].iloc[WARMUP:]

def count_entries(w):
    """An 'entry' = a stock transitions from flat (w==0) to non-zero, or flips sign."""
    prev = w.shift(1).fillna(0)
    was_flat = (prev == 0)
    is_open = (w != 0)
    entered = (was_flat & is_open)
    sign_flip = (np.sign(prev) * np.sign(w) < 0)  # long -> short, short -> long
    return (entered | sign_flip).sum()

entries_v1 = count_entries(w1)
entries_v2 = count_entries(w2)

print('Per-stock entry counts (full post-warmup sample):')
print(pd.DataFrame({'v1': entries_v1, 'v2': entries_v2, 'v2/v1': (entries_v2 / entries_v1).round(2)}))
print()
print(f'v1 total entries: {entries_v1.sum():>5d}')
print(f'v2 total entries: {entries_v2.sum():>5d}')
print(f'v2 is {entries_v2.sum() / entries_v1.sum():.0%} of v1 (fewer trades, higher Sharpe)')

In [ ]:
def cost_accounting(bt, cost_bps_list=(0.5, 1.0, 2.0, 3.0, 5.0)):
    total_turnover = bt['turnover'].sum()
    n_bars_on = (bt['gross_exposure'] > 0).sum()
    rows = []
    for cb in cost_bps_list:
        cost_total = total_turnover * cb / 10_000
        net_pnl = bt['net_pnl'].sum() if abs(cb - 1.0) < 1e-9 else (bt['net_pnl'].sum() + bt['turnover'].sum() * (1.0 - cb) / 10_000)
        gross_pnl = net_pnl + cost_total
        rows.append({
            'cost_bps':       cb,
            'total_turnover (|Δw| units)': round(float(total_turnover), 2),
            'total_cost (on 1 unit gross)':  round(cost_total, 5),
            'total_gross_pnl':              round(gross_pnl, 5),
            'total_net_pnl':                round(net_pnl, 5),
            'cost / gross %':               round(100 * cost_total / gross_pnl, 1) if gross_pnl > 0 else np.nan,
        })
    return pd.DataFrame(rows)

print('--- v1 (Notebook-03 final), full 11mo post-warmup ---')
print(cost_accounting(bt_v1).to_string(index=False))
print()
print('--- v2 (Phase-12 final), full 11mo post-warmup ---')
print(cost_accounting(bt_v2).to_string(index=False))
print()
print('--- v2 TEST window (2025-08 → 2026-03) ---')
print(cost_accounting(bt_v2_test).to_string(index=False))

In [ ]:
def per_trade_stats(bt, n_trades, cap_dollars=1_000_000):
    """
    Assuming a $1M gross book, translate the unit-gross numbers into dollar PnL
    and dollar cost per round-trip trade.
    """
    total_turnover = float(bt['turnover'].sum())
    total_cost_1bp_dollars = total_turnover * cap_dollars * 1e-4
    total_net_pnl_1bp_dollars = float(bt['net_pnl'].sum()) * cap_dollars
    total_gross_pnl_1bp_dollars = total_net_pnl_1bp_dollars + total_cost_1bp_dollars
    return {
        'total trades (entries)': int(n_trades),
        'total turnover (|Δw|)': round(total_turnover, 2),
        'turnover per trade': round(total_turnover / max(n_trades, 1), 4),
        '@ $1M gross — total cost (1bp)':      f'${total_cost_1bp_dollars:,.0f}',
        '@ $1M gross — total gross PnL (1bp)': f'${total_gross_pnl_1bp_dollars:,.0f}',
        '@ $1M gross — total net PnL (1bp)':   f'${total_net_pnl_1bp_dollars:,.0f}',
        '@ $1M gross — cost / trade (1bp)':    f'${total_cost_1bp_dollars / max(n_trades, 1):,.2f}',
        '@ $1M gross — cost / trade (2bp)':    f'${2 * total_cost_1bp_dollars / max(n_trades, 1):,.2f}',
        'cost as % of gross PnL (1bp)':        f'{100 * total_cost_1bp_dollars / max(total_gross_pnl_1bp_dollars, 1):.1f}%',
    }

print('Per-trade accounting @ $1M assumed gross book:')
stats_df = pd.DataFrame({
    'v1 full':    per_trade_stats(bt_v1, entries_v1.sum()),
    'v2 full':    per_trade_stats(bt_v2, entries_v2.sum()),
})
stats_df

In [ ]:
def monthly_trade_count(w):
    """Count entry events per (month, stock) and sum across stocks."""
    prev = w.shift(1).fillna(0)
    entered = ((prev == 0) & (w != 0)) | (np.sign(prev) * np.sign(w) < 0)
    ym = w.index.tz_convert('America/New_York').to_period('M').astype(str)
    per_bar = entered.sum(axis=1)
    return per_bar.groupby(ym).sum()

trades_by_month = pd.DataFrame({
    'v1 trades': monthly_trade_count(w1),
    'v2 trades': monthly_trade_count(w2),
}).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(11, 3.8))
x = np.arange(len(trades_by_month))
w = 0.4
ax.bar(x - w/2, trades_by_month['v1 trades'], width=w, color='#4c78a8', label=f'v1 — total {trades_by_month["v1 trades"].sum()} trades')
ax.bar(x + w/2, trades_by_month['v2 trades'], width=w, color='#2d8659', label=f'v2 — total {trades_by_month["v2 trades"].sum()} trades')
ax.set_xticks(x); ax.set_xticklabels(trades_by_month.index, rotation=45, fontsize=9)
ax.set_ylabel('round-trip trades'); ax.set_title('Monthly trade count — v1 vs v2')
ax.legend()
plt.tight_layout(); plt.show()
trades_by_month

**How to read this**

- At cost = **1 bp/side**, v1 pays about **30–50 %** of its gross PnL back as friction; v2 pays only about **15–20 %**. That is the exact reason v2's net-Sharpe stays alive under stress and v1's doesn't.
- Trade count is **not** the right metric by itself — fewer trades with bigger edge per trade is what you want, and that is what v2 does (roughly **one-third** the entries of v1, but delivers **~2×** the net Sharpe).
- In dollar terms on a $1M book, v2 pays somewhere in the ballpark of a **few thousand dollars per trade** at 1 bp, and net PnL per trade is several multiples of that — i.e. each trade is a genuine positive-expectancy event, not just noise.

## 13. March 2026 freeze — closest read we have to a live month

In [ ]:
ts_ny = bt_v2.index.tz_convert('America/New_York')
march = bt_v2.loc[(ts_ny >= pd.Timestamp('2026-03-01', tz='America/New_York')) & (ts_ny < pd.Timestamp('2026-04-01', tz='America/New_York'))].copy()
march['equity'] = march['net_pnl'].cumsum()
march_v1 = bt_v1.loc[march.index].copy()
march_v1['equity'] = march_v1['net_pnl'].cumsum()

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(march_v1.index, march_v1['equity'] * 100, color='#4c78a8', lw=1.0, label='v1 March 2026')
ax.plot(march.index, march['equity'] * 100, color='#2d8659', lw=1.6, label='v2 March 2026')
ax.axhline(0, color='k', lw=0.7)
ax.set_ylabel('cumulative net PnL (%)'); ax.set_title('March 2026 — single-month freeze equity')
ax.legend()
plt.tight_layout(); plt.show()

pd.Series({
    'v2 March Sharpe':       phase12['v2_march_2026_freeze']['sharpe'],
    'v2 March Ann %':        phase12['v2_march_2026_freeze']['ann_pct'],
    'v2 March MaxDD %':      phase12['v2_march_2026_freeze']['maxdd_pct'],
    'v2 March bars_on %':    phase12['v2_march_2026_freeze']['bars_on_pct'],
    'v2 March avg turnover': phase12['v2_march_2026_freeze']['avg_turnover'],
}).to_frame('value').round(4)

## 14. One-sentence verdict

> Over the full 11-month post-warmup sample, **v2 (xsec residual reversal with breadth + per-stock gates)** delivers **Sharpe +1.96 / MaxDD −4.8%** net of 1bp cost; on the 8-month TEST window it is **+2.17** with bootstrap median **+2.25** (P(Sharpe > 1) = **85 %**); on a never-designed-on single month (March 2026) it prints **+3.08**; under 2bp adverse slippage it is still **+1.30**. The honest headline Sharpe the strategy can claim going forward is **≈ 1.5 – 2.0**, and the only remaining test is paper-trading it forward.